# RAG Chatbot for Research Papers

A retrieval-augmented generation system for querying academic papers using:
- **PyMuPDF** for PDF extraction (handles two-column layouts)
- **FAISS** for vector storage
- **sentence-transformers** for embeddings
- **Claude (Anthropic)** for answer generation
- **Hybrid retrieval**: Vector search + BM25 + cross-encoder re-ranking

## Usage
1. **Phase 1** (run once): Execute cells 1-6 to ingest PDFs and build the index
2. **Phase 2** (every session): Execute cells 1, 7-10 to load index and chat

---
## Phase 1: Setup & Ingestion (Run Once)
---

### Cell 1: Imports & Configuration

In [1]:
# Uncomment to install dependencies (first time only):
!pip install pymupdf faiss-cpu sentence-transformers anthropic rank-bm25 tqdm

import os
import json
import re
import fitz  # PyMuPDF
import faiss
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer, CrossEncoder
from anthropic import Anthropic
from tqdm.notebook import tqdm
from rank_bm25 import BM25Okapi

# ============================================================
# CONFIGURATION
# ============================================================

PAPERS_DIR = Path("./Reference Papers")    # folder of PDFs
VECTOR_STORE_DIR = Path("./vector_store")  # where index gets saved
EMBEDDING_MODEL = "all-MiniLM-L6-v2"      # 384-dim, fast
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-12-v2"  # precision boost
CHUNK_SIZE = 768          # tokens approx (chars / 4 as rough proxy)
CHUNK_OVERLAP = 96        # overlap in tokens approx
TOP_K = 8                 # final chunks to return after re-ranking
RERANK_POOL = 24          # candidates to pull before re-ranking

# --- Model toggle ---
# Sonnet: fast, cheap, good for grounded Q&A over retrieved context
# Opus:   deeper reasoning, use if answers feel shallow
# CLAUDE_MODEL = "claude-sonnet-4-20250514"
# #CLAUDE_MODEL = "claude-3-5-sonnet-20241022"
# # CLAUDE_MODEL = "claude-opus-4-20250514"  # uncomment to switch
# --- Model toggle ---
CLAUDE_MODEL = "claude-sonnet-4-6"
# CLAUDE_MODEL = "claude-opus-4-6"  # uncomment to switch

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_API_KEY:
    from getpass import getpass
    ANTHROPIC_API_KEY = getpass("Enter your Anthropic API key: ")

print("=" * 60)
print("Configuration loaded")
print(f"Papers directory: {PAPERS_DIR}")
print(f"Vector store: {VECTOR_STORE_DIR}")
print(f"Claude model: {CLAUDE_MODEL}")
print("=" * 60)

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


/Users/christopherwaight/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Configuration loaded
Papers directory: Reference Papers
Vector store: vector_store
Claude model: claude-sonnet-4-6


### Cell 2: PDF Extraction Function

In [2]:
def extract_pdf(pdf_path: Path) -> dict:
    """
    Extract text and structural info from a PDF.
    
    Returns:
        {
            "filename": str,
            "title": str (best guess),
            "sections": [{"heading": str or None, "text": str}, ...]
        }
    """
    doc = fitz.open(pdf_path)
    
    # === Extract title from first page ===
    title = pdf_path.stem  # fallback to filename
    try:
        page1 = doc[0]
        blocks = page1.get_text("dict", sort=True)["blocks"]
        # Find largest font size text on page 1 (likely the title)
        max_font_size = 0
        for block in blocks:
            if "lines" in block:
                for line in block["lines"]:
                    for span in line["spans"]:
                        if span["size"] > max_font_size:
                            max_font_size = span["size"]
                            title = span["text"].strip()
    except Exception:
        pass  # keep filename as title
    
    # === Extract sections ===
    sections = []
    current_heading = None
    current_text = []
    
    stop_extraction = False  # flag to stop at References
    
    for page_num in range(len(doc)):
        if stop_extraction:
            break
            
        page = doc[page_num]
        
        # Get text with sort=True to handle two-column layouts
        try:
            page_dict = page.get_text("dict", sort=True)
            blocks = page_dict["blocks"]
        except Exception:
            # Fallback to simple text extraction if dict fails
            text = page.get_text("text", sort=True)
            if text.strip():
                current_text.append(text)
            continue
        
        # Calculate median font size for heading detection
        font_sizes = []
        for block in blocks:
            if "lines" in block:
                for line in block["lines"]:
                    for span in line["spans"]:
                        font_sizes.append(span["size"])
        median_font = np.median(font_sizes) if font_sizes else 10
        
        # Process blocks
        for block in blocks:
            if "lines" not in block:
                continue
                
            block_text = ""
            block_font_size = 0
            
            for line in block["lines"]:
                line_text = ""
                for span in line["spans"]:
                    line_text += span["text"]
                    block_font_size = max(block_font_size, span["size"])
                block_text += line_text + " "
            
            block_text = block_text.strip()
            if not block_text:
                continue
            
            # Check if this is the References section (stop extraction)
            if re.match(r"^(References|Bibliography|REFERENCES|BIBLIOGRAPHY)\s*$", block_text):
                stop_extraction = True
                break
            
            # Detect if this block is a heading
            is_heading = False
            
            # Heuristic 1: Larger font than median
            if block_font_size > median_font * 1.1:
                is_heading = True
            
            # Heuristic 2: Short line with common heading patterns
            if len(block_text) < 100 and (
                re.match(r"^[IVX]+\.", block_text) or  # Roman numerals
                re.match(r"^\d+(\.\d+)*\.?\s+[A-Z]", block_text) or  # Numbered sections
                block_text.isupper() or  # All caps
                block_text in ["Abstract", "Introduction", "Conclusion", "Methods", "Results", "Discussion"]
            ):
                is_heading = True
            
            if is_heading:
                # Save previous section
                if current_text:
                    sections.append({
                        "heading": current_heading,
                        "text": " ".join(current_text)
                    })
                current_heading = block_text
                current_text = []
            else:
                current_text.append(block_text)
    
    # Add final section
    if current_text:
        sections.append({
            "heading": current_heading,
            "text": " ".join(current_text)
        })
    
    doc.close()
    
    # === Extract abstract (first section whose heading matches "Abstract") ===
    abstract = ""
    for sec in sections[:4]:  # only look in first few sections
        heading = (sec.get("heading") or "").strip().lower()
        if heading in ("abstract", "i. abstract", "1. abstract"):
            abstract = re.sub(r"\s+", " ", sec["text"]).strip()[:500]
            break

    return {
        "filename": pdf_path.name,
        "title": title,
        "abstract": abstract,
        "sections": sections
    }

print("PDF extraction function defined")

PDF extraction function defined


### Cell 3: Section-Aware Chunking

In [3]:
def chunk_document(doc: dict, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP) -> list:
    """
    Chunk a document respecting section boundaries.
    
    Returns:
        [{
            "text": str,           # the chunk content
            "filename": str,       # source PDF
            "title": str,          # paper title
            "section": str or None # heading this chunk falls under
        }, ...]
    """
    chunks = []
    chunk_size_chars = chunk_size * 4  # rough token-to-char conversion
    overlap_chars = overlap * 4
    
    for section in doc["sections"]:
        section_text = section["text"]
        section_heading = section["heading"]
        
        # Clean up whitespace
        section_text = re.sub(r"\s+", " ", section_text).strip()
        
        if len(section_text) < 50:
            continue  # Skip very short sections
        
        # If section fits in one chunk, keep it whole
        if len(section_text) <= chunk_size_chars:
            # Add metadata header
            abstract_line = f"[Abstract: {doc.get('abstract', '')[:300]}]\n" if doc.get('abstract') else ""
            chunk_text = f"[Title: {doc['title']} | Section: {section_heading or 'N/A'}]\n{abstract_line}{section_text}"
            chunks.append({
                "text": chunk_text,
                "filename": doc["filename"],
                "title": doc["title"],
                "section": section_heading
            })
        else:
            # Split at sentence boundaries
            sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', section_text)
            
            current_chunk = ""
            for sentence in sentences:
                if len(current_chunk) + len(sentence) <= chunk_size_chars:
                    current_chunk += sentence + " "
                else:
                    # Save current chunk
                    if current_chunk.strip():
                        abstract_line = f"[Abstract: {doc.get('abstract', '')[:300]}]\n" if doc.get('abstract') else ""
                        chunk_text = f"[Title: {doc['title']} | Section: {section_heading or 'N/A'}]\n{abstract_line}{current_chunk.strip()}"
                        chunks.append({
                            "text": chunk_text,
                            "filename": doc["filename"],
                            "title": doc["title"],
                            "section": section_heading
                        })
                    
                    # Start new chunk with overlap
                    if len(current_chunk) > overlap_chars:
                        current_chunk = current_chunk[-overlap_chars:] + sentence + " "
                    else:
                        current_chunk = sentence + " "
            
            # Add final chunk
            if current_chunk.strip():
                abstract_line = f"[Abstract: {doc.get('abstract', '')[:300]}]\n" if doc.get('abstract') else ""
                chunk_text = f"[Title: {doc['title']} | Section: {section_heading or 'N/A'}]\n{abstract_line}{current_chunk.strip()}"
                chunks.append({
                    "text": chunk_text,
                    "filename": doc["filename"],
                    "title": doc["title"],
                    "section": section_heading
                })
    
    return chunks

print("Chunking function defined")

Chunking function defined


### Cell 4: Build Embeddings & FAISS Index

In [4]:
def build_index(all_chunks: list) -> tuple:
    """
    Embed all chunks and build a FAISS index.
    
    Returns:
        (index, all_chunks): FAISS index and aligned chunks list
    """
    print("Loading embedding model...")
    model = SentenceTransformer(EMBEDDING_MODEL)
    
    print(f"Embedding {len(all_chunks)} chunks...")
    texts = [c["text"] for c in all_chunks]
    embeddings = model.encode(
        texts, 
        show_progress_bar=True, 
        normalize_embeddings=True
    )
    
    print("Building FAISS index...")
    index = faiss.IndexFlatIP(embeddings.shape[1])  # Inner product (cosine similarity)
    index.add(embeddings.astype("float32"))
    
    print(f"Index built with {index.ntotal} vectors")
    return index, all_chunks

print("Index building function defined")

Index building function defined


### Cell 5: Save Index to Disk

In [5]:
def save_index(index, chunks, store_dir=VECTOR_STORE_DIR):
    """
    Save FAISS index and chunk metadata to disk.
    """
    store_dir.mkdir(exist_ok=True)
    
    faiss.write_index(index, str(store_dir / "faiss_index.bin"))
    
    with open(store_dir / "chunks_metadata.json", "w") as f:
        json.dump(chunks, f, indent=2)
    
    print("="*60)
    print(f"Saved {index.ntotal} vectors and metadata to {store_dir}")
    print("="*60)

print("Save function defined")

Save function defined


### Cell 6: Run Full Ingestion Pipeline

**Run this cell once to process all PDFs and build the index.**  
After this completes, you can skip directly to Cell 7 in future sessions.

In [6]:
print("="*60)
print("Starting PDF ingestion...")
print("="*60)

# Find all PDFs
pdf_files = sorted(PAPERS_DIR.glob("*.pdf"))
print(f"\nFound {len(pdf_files)} PDFs in {PAPERS_DIR}\n")

# Extract and chunk all PDFs
all_chunks = []
failed_pdfs = []

for pdf_path in tqdm(pdf_files, desc="Extracting PDFs"):
    try:
        doc = extract_pdf(pdf_path)
        chunks = chunk_document(doc)
        all_chunks.extend(chunks)
        print(f"  ✓ {pdf_path.name}: {len(chunks)} chunks")
    except Exception as e:
        failed_pdfs.append(pdf_path.name)
        print(f"  ✗ FAILED {pdf_path.name}: {e}")

print("\n" + "-"*60)
print(f"Total chunks extracted: {len(all_chunks)}")
if failed_pdfs:
    print(f"Failed PDFs ({len(failed_pdfs)}): {', '.join(failed_pdfs)}")
print("-"*60 + "\n")

# Build and save index
if all_chunks:
    index, chunks = build_index(all_chunks)
    save_index(index, chunks)
    print("\n✓ Ingestion complete! You can now skip to Cell 7 in future sessions.")
else:
    print("\n✗ No chunks extracted. Check your PDF directory and try again.")

Starting PDF ingestion...

Found 128 PDFs in Reference Papers



Extracting PDFs:   0%|          | 0/128 [00:00<?, ?it/s]

  ✓ [100] Vector Field-based Collision Avoidance for - Field.pdf: 23 chunks
  ✓ [101] 093904_1_5.0268535.pdf: 51 chunks
  ✓ [102] Rapid Deterministic Wave Prediction Using a Sparse Array of Buoys.pdf: 37 chunks
  ✓ [103] Cooperative Heterogeneous Robot Team for Planetary Exploration Using Deep RL.pdf: 43 chunks
  ✓ [104] 1-s2.0-S0167278905004446-main.pdf: 54 chunks
  ✓ [105] 19970018786.pdf: 28 chunks
  ✓ [106] Deep Sea Research II - doi 10.1016 j.dsr2.2008.08.008.pdf: 36 chunks
  ✓ [107] 30-2_hanson.pdf: 17 chunks
  ✓ [108] Odor Detection - Robotics and Autonomous Systems 2019.pdf: 51 chunks
  ✓ [109] A_Path_Planning_Approach_for_Multi-AUV_Systems_With_Concurrent_Stationary_Node_Access_and_Adaptive_Sampling.pdf: 30 chunks
  ✓ [10] Coordinated Sampling of Dynamic Oceanographic Features - Das - 2012.pdf: 45 chunks
  ✓ [110] A_Survey_of_Intelligent_Mapless_Multi-Robot_Systems_Perception_Distributed_Planning_and_Predictive_Cooperation.pdf: 32 chunks
  ✓ [111] Dynamic Rotation and Stretch 

Batches:   0%|          | 0/143 [00:00<?, ?it/s]

Building FAISS index...
Index built with 4550 vectors
Saved 4550 vectors and metadata to vector_store

✓ Ingestion complete! You can now skip to Cell 7 in future sessions.


---
## Phase 2: Load & Chat (Every Session)
---

### Cell 7: Load Saved Index

In [7]:
def load_index(store_dir=VECTOR_STORE_DIR):
    """
    Load FAISS index and chunk metadata from disk.
    """
    index = faiss.read_index(str(store_dir / "faiss_index.bin"))
    
    with open(store_dir / "chunks_metadata.json") as f:
        chunks = json.load(f)
    
    print(f"Loaded {index.ntotal} vectors from {store_dir}")
    return index, chunks

print("="*60)
print("Loading models and index...")
print("="*60)

# Load saved index
index, chunks = load_index()

# Load embedding model
print("Loading embedding model...")
embed_model = SentenceTransformer(EMBEDDING_MODEL)

# Load re-ranker
print("Loading re-ranker...")
reranker = CrossEncoder(RERANKER_MODEL)

# Build BM25 index (fast, runs from loaded chunks)
print("Building BM25 index...")
tokenized_corpus = [c["text"].lower().split() for c in chunks]
bm25 = BM25Okapi(tokenized_corpus)

print("="*60)
print(f"BM25 index built over {len(chunks)} chunks")
print("All models loaded — ready to chat!")
print("="*60)

Loading models and index...
Loaded 4550 vectors from vector_store
Loading embedding model...
Loading re-ranker...
Building BM25 index...
BM25 index built over 4550 chunks
All models loaded — ready to chat!


### Cell 8: Retrieval Function (Hybrid + Re-ranking)

In [8]:
def retrieve(query: str, index, chunks, embed_model, bm25, reranker, 
             pool_size=RERANK_POOL, top_k=TOP_K) -> list:
    """
    Hybrid retrieval + cross-encoder re-ranking.
    
    Stage 1: Vector + BM25 → RRF merge → pool_size candidates
    Stage 2: Cross-encoder re-ranks candidates → top_k results
    
    Returns:
        List of top_k chunks with rerank_score added
    """
    # === Stage 1: Vector arm ===
    query_vec = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    vec_scores, vec_indices = index.search(query_vec, pool_size)
    vec_results = [(int(idx), float(score)) for score, idx in zip(vec_scores[0], vec_indices[0])]
    
    # === Stage 1: BM25 arm ===
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_top = np.argsort(bm25_scores)[-pool_size:][::-1]
    bm25_results = [(int(idx), float(bm25_scores[idx])) for idx in bm25_top]
    
    # === Stage 1: Reciprocal Rank Fusion ===
    rrf_k = 60
    fused = {}
    
    for rank, (idx, _) in enumerate(vec_results):
        fused[idx] = fused.get(idx, 0) + 1 / (rrf_k + rank + 1)
    
    for rank, (idx, _) in enumerate(bm25_results):
        fused[idx] = fused.get(idx, 0) + 1 / (rrf_k + rank + 1)
    
    # Take top pool_size candidates for re-ranking
    candidates = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:pool_size]
    
    # === Stage 2: Cross-encoder re-ranking ===
    pairs = [(query, chunks[idx]["text"]) for idx, _ in candidates]
    rerank_scores = reranker.predict(pairs)
    
    # Combine and sort by re-ranker score
    scored = list(zip([idx for idx, _ in candidates], rerank_scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    
    # Build results
    results = []
    for idx, score in scored[:top_k]:
        chunk = chunks[idx].copy()
        chunk["rerank_score"] = float(score)
        results.append(chunk)
    
    return results

print("Retrieval function defined")

Retrieval function defined


### Cell 9: Claude Generation Function

In [9]:
client = Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """You are a research assistant answering questions based on a corpus of academic papers.
You will be given retrieved passages from the papers as context.

Rules:
- Answer based ONLY on the provided context. If the context doesn't contain enough information, say so.
- Cite your sources by mentioning the paper title and section when you use information from a passage.
- Be precise and technical. The user is a researcher.
- If multiple papers address the question differently, summarize the range of perspectives."""

def ask(query: str, context_chunks: list, conversation_history: list) -> str:
    """
    Generate an answer using Claude based on retrieved context.
    
    Args:
        query: User question
        context_chunks: Retrieved chunks from hybrid retrieval
        conversation_history: List of {"role": "user"/"assistant", "content": str}
    
    Returns:
        Claude's answer as a string
    """
    # Format retrieved context
    context_str = "\n\n---\n\n".join(
        f"[Source: {c['title']} | Section: {c.get('section', 'N/A')} | File: {c['filename']}]\n{c['text']}"
        for c in context_chunks
    )
    
    user_message = f"""<context>
{context_str}
</context>

Question: {query}"""
    
    messages = conversation_history + [{"role": "user", "content": user_message}]
    
    response = client.messages.create(
        model=CLAUDE_MODEL,
        max_tokens=2048,
        system=SYSTEM_PROMPT,
        messages=messages,
    )
    
    return response.content[0].text

print("Claude generation function defined")

Claude generation function defined


### Cell 10: Interactive Chat Loop

In [10]:
def chat():
    """
    Simple chat loop.
    
    Commands:
        'quit' - exit
        'clear' - reset conversation history
    """
    history = []
    
    print("="*60)
    print("RAG Chatbot ready. Ask questions about your papers.")
    print("Commands: 'quit' to exit, 'clear' to reset conversation")
    print("="*60)
    print()
    
    while True:
        query = input("You: ").strip()
        
        if not query:
            continue
        
        if query.lower() == "quit":
            print("Goodbye!")
            break
        
        if query.lower() == "clear":
            history = []
            print("Conversation cleared.\n")
            continue
        
        # Retrieve relevant chunks
        print("\n[Retrieving relevant passages...]")
        results = retrieve(query, index, chunks, embed_model, bm25, reranker)
        
        # Generate answer
        print("[Generating answer...]\n")
        answer = ask(query, results, history)
        
        # Update history (keep last 12 messages = 6 turns)
        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": answer})
        if len(history) > 12:
            history = history[-12:]
        
        # Display answer
        print("-"*60)
        print(f"Claude: {answer}")
        print("-"*60)
        
        # Show sources
        unique_sources = sorted(set(r['filename'] for r in results))
        print(f"\n[Retrieved from {len(results)} chunks across {len(unique_sources)} papers:]")
        for source in unique_sources:
            print(f"  • {source}")
        print()

# Start the chat
chat()

RAG Chatbot ready. Ask questions about your papers.
Commands: 'quit' to exit, 'clear' to reset conversation


[Retrieving relevant passages...]
[Generating answer...]

------------------------------------------------------------
Claude: Based on the provided context from Mateus et al. (2020), the answer is **yes, buoyant (resurfaced) bodies are considerably easier to search for**, though significant challenges remain. Here is a detailed breakdown:

## When Bodies Are Submerged (Stages 1–2)
During the **Post-Mortem Submersion Interval (PMSI)**, the body sinks and may be trapped at the bottom by:
- Natural features (rocks, caves, branches)
- Heavy objects (clothes, stones)
- Depth making bloating insufficient to provide buoyancy

At this stage, active surface-based search methods are **unsuitable**, and modeling drift at the surface is irrelevant. The body is effectively hidden and difficult to locate.

## When Bodies Resurface (Stage 3)
Once decomposition induces bloating and the body r

In [ ]:
pip install --upgrade anthropic

---
## Cell 11: Testing & Examples
---

### Quick Retrieval Test

Test the retrieval system without the full chat loop:

In [ ]:
# Test query
test_query = "gradient estimation multirobot formations"

print(f"Query: {test_query}\n")
print("="*60)

results = retrieve(test_query, index, chunks, embed_model, bm25, reranker, top_k=5)

for i, result in enumerate(results, 1):
    print(f"\nResult {i} (score: {result['rerank_score']:.4f})")
    print(f"Title: {result['title']}")
    print(f"Section: {result.get('section', 'N/A')}")
    print(f"File: {result['filename']}")
    print(f"Text preview: {result['text'][:200]}...")
    print("-"*60)

### Example Queries to Try

Based on your research papers, here are some suggested queries:

1. **Gradient estimation**: `"How do multirobot systems estimate gradients using formations?"`
2. **Vector field navigation**: `"What methods are used for vector field navigation with robot clusters?"`
3. **Source seeking**: `"What are the main approaches to cooperative source seeking?"`
4. **Formation control**: `"How is formation control achieved in multirobot systems?"`
5. **Critical points**: `"How can robots detect critical points in scalar or vector fields?"`
6. **Experimental validation**: `"What testbeds have been used for multirobot field navigation?"`

Try these in the chat interface above!

### Inspect Index Statistics

In [ ]:
print("="*60)
print("INDEX STATISTICS")
print("="*60)
print(f"Total chunks: {len(chunks)}")
print(f"Total vectors in FAISS: {index.ntotal}")
print(f"Vector dimension: {index.d}")
print()

# Unique papers
unique_papers = set(c['filename'] for c in chunks)
print(f"Unique papers: {len(unique_papers)}")
print()

# Chunks per paper distribution
from collections import Counter
paper_counts = Counter(c['filename'] for c in chunks)
print("Top 10 papers by chunk count:")
for paper, count in paper_counts.most_common(10):
    print(f"  {count:4d} chunks - {paper}")
print()

# Section distribution
sections = [c.get('section', 'N/A') for c in chunks]
section_counts = Counter(sections)
print(f"Unique sections: {len(section_counts)}")
print("\nTop 10 sections:")
for section, count in section_counts.most_common(10):
    section_name = section if section and len(section) < 50 else (section[:47] + '...' if section else 'N/A')
    print(f"  {count:4d} chunks - {section_name}")
print("="*60)